In [1]:
# 목적: 고유 리스크 수집 + 동적 자산 배분

import pandas as pd
import numpy as np
import yfinance as yf
import os
import matplotlib.pyplot as plt
import requests

data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")

# 저장된 데이터 로드
copula_scores = pd.read_csv(f"{data_path}/copula_risk_scores.csv",
                             index_col="Date", parse_dates=True)

TICKERS = ["AAPL", "MSFT", "GOOGL", "JPM", "SOXX"]

print("로드 완료")
print(f"Copula 리스크 점수: {copula_scores.shape}")

로드 완료
Copula 리스크 점수: (2102, 5)


In [2]:
# Step 1. 애널리스트 등급 변화 수집
def get_analyst_risk(ticker):
    """
    애널리스트 등급 변화로 고유 리스크 점수 계산
    - Downgrade 비율이 높을수록 리스크 높음
    """
    try:
        stock = yf.Ticker(ticker)
        rec = stock.recommendations

        if rec is None or rec.empty:
            print(f"  {ticker}: 등급 데이터 없음 → 0.5 기본값")
            return 0.5

        # 최근 3개월 등급만 사용
        recent = rec.tail(20)

        # 등급을 숫자로 변환
        grade_map = {
            "Strong Buy": -1.0, "Buy": -0.5, "Outperform": -0.3,
            "Neutral": 0.0,  "Hold": 0.0,  "Market Perform": 0.0,
            "Underperform": 0.5, "Sell": 0.8, "Strong Sell": 1.0
        }

        scores = []
        for _, row in recent.iterrows():
            to_grade = str(row.get("To Grade", "Neutral"))
            from_grade = str(row.get("From Grade", "Neutral"))

            to_score   = grade_map.get(to_grade, 0.0)
            from_score = grade_map.get(from_grade, 0.0)

            # 등급 악화면 양수, 개선이면 음수
            scores.append(to_score - from_score)

        if not scores:
            return 0.5

        # -1 ~ 1 범위를 0 ~ 1 범위로 변환
        avg = np.mean(scores)
        return round(max(0, min(1, (avg + 1) / 2)), 4)

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

# 전체 종목 수집
print("애널리스트 등급 수집 중...")
analyst_scores = {}
for ticker in TICKERS:
    score = get_analyst_risk(ticker)
    analyst_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'위험' if score > 0.5 else '안전'})")

애널리스트 등급 수집 중...
  AAPL: 0.5000 (안전)
  MSFT: 0.5000 (안전)
  GOOGL: 0.5000 (안전)
  JPM: 0.5000 (안전)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: 등급 데이터 없음 → 0.5 기본값
  SOXX: 0.5000 (안전)


In [3]:
stock = yf.Ticker("AAPL")
rec = stock.recommendations
print(type(rec))
print(rec)

<class 'pandas.DataFrame'>
  period  strongBuy  buy  hold  sell  strongSell
0     0m          6   22    16     1           2
1    -1m          6   22    16     1           2
2    -2m          7   23    15     1           2
3    -3m          7   25    14     1           1


In [4]:
# 실제 데이터 형식에 맞게 수정
def get_analyst_risk(ticker):
    """
    애널리스트 등급 집계로 고유 리스크 점수 계산
    - Sell/StrongSell 비율이 높을수록 리스크 높음
    """
    try:
        stock = yf.Ticker(ticker)
        rec = stock.recommendations

        if rec is None or rec.empty:
            print(f"  {ticker}: 데이터 없음 → 0.5 기본값")
            return 0.5

        # 최근 1개월 (0m) 데이터만 사용
        latest = rec[rec["period"] == "0m"].iloc[0]

        strong_buy  = latest["strongBuy"]
        buy         = latest["buy"]
        hold        = latest["hold"]
        sell        = latest["sell"]
        strong_sell = latest["strongSell"]
        total       = strong_buy + buy + hold + sell + strong_sell

        if total == 0:
            return 0.5

        # 가중 평균: strongBuy=0, buy=0.25, hold=0.5, sell=0.75, strongSell=1.0
        weighted = (
            strong_buy  * 0.0  +
            buy         * 0.25 +
            hold        * 0.5  +
            sell        * 0.75 +
            strong_sell * 1.0
        ) / total

        return round(weighted, 4)

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

# 전체 종목 수집
print("애널리스트 등급 수집 중...")
analyst_scores = {}
for ticker in TICKERS:
    score = get_analyst_risk(ticker)
    analyst_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'위험' if score > 0.5 else '안전'})")

애널리스트 등급 수집 중...
  AAPL: 0.3457 (안전)
  MSFT: 0.2098 (안전)
  GOOGL: 0.2227 (안전)
  JPM: 0.3333 (안전)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: 데이터 없음 → 0.5 기본값
  SOXX: 0.5000 (안전)


In [5]:
# Step 2. 어닝 서프라이즈 수집
def get_earnings_risk(ticker):
    """
    최근 어닝 서프라이즈로 고유 리스크 점수 계산
    - 어닝 미달(miss)이 많을수록 리스크 높음
    """
    try:
        stock = yf.Ticker(ticker)
        earnings = stock.earnings_dates

        if earnings is None or earnings.empty:
            print(f"  {ticker}: 어닝 데이터 없음 → 0.5 기본값")
            return 0.5

        # EPS Surprise % 컬럼 확인
        if "EPS Surprise %" not in earnings.columns:
            print(f"  {ticker}: EPS Surprise 컬럼 없음 → 0.5 기본값")
            return 0.5

        # 최근 4분기만 사용
        recent = earnings.dropna(subset=["EPS Surprise %"]).head(4)

        if recent.empty:
            return 0.5

        # 서프라이즈 % → 리스크 점수
        # 양수(beat) → 낮은 리스크, 음수(miss) → 높은 리스크
        surprises = recent["EPS Surprise %"].values
        avg_surprise = np.mean(surprises)

        # -50% ~ +50% 범위를 0~1로 정규화 (miss일수록 1에 가깝게)
        score = max(0, min(1, 0.5 - avg_surprise / 100))

        return round(score, 4)

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

# 전체 종목 수집
print("어닝 서프라이즈 수집 중...")
earnings_scores = {}
for ticker in TICKERS:
    score = get_earnings_risk(ticker)
    earnings_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'miss 위험' if score > 0.5 else 'beat 안정'})")

어닝 서프라이즈 수집 중...
  AAPL: EPS Surprise 컬럼 없음 → 0.5 기본값
  AAPL: 0.5000 (beat 안정)
  MSFT: EPS Surprise 컬럼 없음 → 0.5 기본값
  MSFT: 0.5000 (beat 안정)
  GOOGL: EPS Surprise 컬럼 없음 → 0.5 기본값
  GOOGL: 0.5000 (beat 안정)
  JPM: EPS Surprise 컬럼 없음 → 0.5 기본값
  JPM: 0.5000 (beat 안정)


SOXX: No earnings dates found, symbol may be delisted


  SOXX: 어닝 데이터 없음 → 0.5 기본값
  SOXX: 0.5000 (beat 안정)


In [6]:
# 실제 컬럼명 확인
stock = yf.Ticker("AAPL")
earnings = stock.earnings_dates
print(earnings.columns.tolist())
print(earnings.head())

['EPS Estimate', 'Reported EPS', 'Surprise(%)']
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-07-30 16:00:00-04:00          1.89           NaN          NaN
2026-04-30 16:00:00-04:00          1.94          2.01         3.46
2026-01-29 16:00:00-05:00          2.67          2.84         6.34
2025-10-30 16:00:00-04:00          1.77          1.85         4.52
2025-07-31 16:00:00-04:00          1.43          1.57        10.12


In [7]:
# 컬럼명 수정
def get_earnings_risk(ticker):
    try:
        stock = yf.Ticker(ticker)
        earnings = stock.earnings_dates

        if earnings is None or earnings.empty:
            return 0.5

        # 최근 4분기 (NaN 제외)
        recent = earnings.dropna(subset=["Surprise(%)"]).head(4)

        if recent.empty:
            return 0.5

        avg_surprise = recent["Surprise(%)"].mean()

        # beat(양수) → 낮은 리스크, miss(음수) → 높은 리스크
        score = max(0, min(1, 0.5 - avg_surprise / 100))

        return round(score, 4)

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

print("어닝 서프라이즈 수집 중...")
earnings_scores = {}
for ticker in TICKERS:
    score = get_earnings_risk(ticker)
    earnings_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'miss 위험' if score > 0.5 else 'beat 안정'})")

SOXX: No earnings dates found, symbol may be delisted


어닝 서프라이즈 수집 중...
  AAPL: 0.4389 (beat 안정)
  MSFT: 0.4207 (beat 안정)
  GOOGL: 0.1721 (beat 안정)
  JPM: 0.4538 (beat 안정)
  SOXX: 0.5000 (beat 안정)


In [8]:
# Step 3. SEC EDGAR 내부자 거래 수집
def get_insider_risk(ticker):
    """
    SEC EDGAR에서 내부자 거래 데이터 수집
    - 내부자 매도가 많을수록 리스크 높음
    """
    try:
        # SEC CIK 번호 조회
        search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&dateRange=custom&startdt=2026-01-01&enddt=2026-07-08&forms=4"
        headers = {"User-Agent": "SentriVaR research@example.com"}
        response = requests.get(search_url, headers=headers, timeout=10)

        if response.status_code != 200:
            return 0.5

        data = response.json()
        hits = data.get("hits", {}).get("hits", [])

        if not hits:
            return 0.5

        # Form 4에서 매수/매도 비율 계산
        buy_count  = 0
        sell_count = 0

        for hit in hits[:20]:  # 최근 20건
            source = hit.get("_source", {})
            form_type = source.get("file_type", "")
            if form_type == "4":
                # 파일명/내용으로 매수/매도 추정
                entity = source.get("entity_name", "").lower()
                if "purchase" in entity or "buy" in entity:
                    buy_count += 1
                else:
                    sell_count += 1

        total = buy_count + sell_count
        if total == 0:
            return 0.5

        # 매도 비율이 높을수록 리스크 높음
        sell_ratio = sell_count / total
        return round(sell_ratio, 4)

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

print("내부자 거래 수집 중...")
insider_scores = {}
for ticker in TICKERS:
    score = get_insider_risk(ticker)
    insider_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'매도 우세' if score > 0.5 else '매수 우세'})")

내부자 거래 수집 중...
  AAPL: 1.0000 (매도 우세)
  MSFT: 1.0000 (매도 우세)
  GOOGL: 1.0000 (매도 우세)
  JPM: 1.0000 (매도 우세)
  SOXX: 0.5000 (매수 우세)


In [9]:
# 실제 SEC 응답 확인
ticker = "AAPL"
search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&dateRange=custom&startdt=2026-01-01&enddt=2026-07-08&forms=4"
headers = {"User-Agent": "SentriVaR research@example.com"}
response = requests.get(search_url, headers=headers, timeout=10)

data = response.json()
hits = data.get("hits", {}).get("hits", [])

print(f"총 {len(hits)}건")
if hits:
    print("\n첫 번째 항목 키:")
    print(hits[0]["_source"].keys())
    print("\n첫 번째 항목 내용:")
    print(hits[0]["_source"])

총 26건

첫 번째 항목 키:
dict_keys(['ciks', 'period_ending', 'file_num', 'display_names', 'xsl', 'schema_version', 'sequence', 'root_forms', 'file_date', 'biz_states', 'sics', 'form', 'adsh', 'film_num', 'biz_locations', 'file_type', 'file_description', 'inc_states', 'items'])

첫 번째 항목 내용:
{'ciks': ['0002100523', '0000320193'], 'period_ending': '2026-05-08', 'file_num': ['001-36743'], 'display_names': ['Borders Ben  (CIK 0002100523)', 'Apple Inc.  (CIK 0000320193)'], 'xsl': 'xslF345X06', 'schema_version': 'X0609', 'sequence': 1, 'root_forms': ['4'], 'file_date': '2026-05-12', 'biz_states': ['CA'], 'sics': ['3571'], 'form': '4', 'adsh': '0001140361-26-020871', 'film_num': ['26970260'], 'biz_locations': ['', 'Cupertino, CA'], 'file_type': '4', 'file_description': 'FORM 4', 'inc_states': ['', 'CA'], 'items': []}


In [10]:
# yfinance insider_transactions로 대체
def get_insider_risk(ticker):
    """
    yfinance 내부자 거래로 고유 리스크 점수 계산
    - 내부자 매도가 많을수록 리스크 높음
    """
    try:
        stock = yf.Ticker(ticker)
        insider = stock.insider_transactions

        if insider is None or insider.empty:
            print(f"  {ticker}: 내부자 거래 데이터 없음 → 0.5 기본값")
            return 0.5

        print(f"  {ticker} 컬럼: {insider.columns.tolist()}")
        print(insider.head(3))
        return 0.5

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

# AAPL만 테스트
get_insider_risk("AAPL")

  AAPL 컬럼: ['Shares', 'Value', 'URL', 'Text', 'Insider', 'Position', 'Transaction', 'Start Date', 'Ownership']
   Shares    Value URL                             Text            Insider  \
0     116  34236.0      Sale at price 295.14 per share.        BORDERS BEN   
1   30104      NaN                                       NEWSTEAD JENNIFER   
2     240      NaN                                             BORDERS BEN   

          Position Transaction Start Date Ownership  
0          Officer             2026-06-16         D  
1  General Counsel             2026-06-15         D  
2          Officer             2026-06-15         D  


0.5

In [11]:
def get_insider_risk(ticker):
    """
    yfinance 내부자 거래로 고유 리스크 점수 계산
    - Sale 비율이 높을수록 리스크 높음
    """
    try:
        stock = yf.Ticker(ticker)
        insider = stock.insider_transactions

        if insider is None or insider.empty:
            return 0.5

        # Text 컬럼에서 Sale/Purchase 파싱
        insider["is_sale"] = insider["Text"].str.contains("Sale", case=False, na=False)
        insider["is_buy"]  = insider["Text"].str.contains("Purchase|Buy", case=False, na=False)

        # Value 기준으로 가중 계산 (금액이 클수록 중요)
        sale_value = insider[insider["is_sale"]]["Value"].fillna(0).sum()
        buy_value  = insider[insider["is_buy"]]["Value"].fillna(0).sum()
        total_value = sale_value + buy_value

        if total_value == 0:
            # 금액 없으면 건수로 계산
            sale_count = insider["is_sale"].sum()
            buy_count  = insider["is_buy"].sum()
            total = sale_count + buy_count
            if total == 0:
                return 0.5
            return round(sale_count / total, 4)

        # 매도 금액 비율이 높을수록 리스크 높음
        return round(sale_value / total_value, 4)

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

print("내부자 거래 수집 중...")
insider_scores = {}
for ticker in TICKERS:
    score = get_insider_risk(ticker)
    insider_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'매도 우세' if score > 0.5 else '매수 우세'})")

내부자 거래 수집 중...
  AAPL: 1.0000 (매도 우세)
  MSFT: 0.9878 (매도 우세)
  GOOGL: 1.0000 (매도 우세)
  JPM: 1.0000 (매도 우세)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: 0.5000 (매수 우세)


In [12]:
def get_insider_risk(ticker):
    """
    내부자 매도 가속도로 리스크 측정
    - 최근 매도가 과거보다 급증하면 리스크 높음
    """
    try:
        stock = yf.Ticker(ticker)
        insider = stock.insider_transactions

        if insider is None or insider.empty:
            return 0.5

        insider["Start Date"] = pd.to_datetime(insider["Start Date"])
        insider["is_sale"] = insider["Text"].str.contains("Sale", case=False, na=False)

        # 날짜 기준 분리
        cutoff = pd.Timestamp.today() - pd.DateOffset(months=3)
        recent = insider[insider["Start Date"] >= cutoff]
        older  = insider[insider["Start Date"] < cutoff]

        recent_sale = recent[recent["is_sale"]]["Value"].fillna(0).sum()
        recent_total = recent["Value"].fillna(0).sum()

        older_sale = older[older["is_sale"]]["Value"].fillna(0).sum()
        older_total = older["Value"].fillna(0).sum()

        recent_ratio = recent_sale / recent_total if recent_total > 0 else 0.5
        older_ratio  = older_sale  / older_total  if older_total  > 0 else 0.5

        # 최근 매도 비율이 과거보다 높으면 리스크 증가
        delta = recent_ratio - older_ratio
        score = max(0, min(1, 0.5 + delta * 0.5))

        return round(score, 4)

    except Exception as e:
        print(f"  {ticker} 에러: {e}")
        return 0.5

print("내부자 거래 수집 중...")
insider_scores = {}
for ticker in TICKERS:
    score = get_insider_risk(ticker)
    insider_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'매도 가속' if score > 0.5 else '매도 안정'})")

내부자 거래 수집 중...
  AAPL: 0.5000 (매도 안정)
  MSFT: 0.5063 (매도 가속)
  GOOGL: 0.2500 (매도 안정)
  JPM: 0.5055 (매도 가속)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: 0.5000 (매도 안정)


In [13]:
# Step 4. 고유 리스크 통합
idiosyncratic_risk = {}

for ticker in TICKERS:
    # 가중치: 어닝(40%) + 애널리스트(30%) + 내부자(30%)
    score = (
        earnings_scores[ticker]  * 0.40 +
        analyst_scores[ticker]   * 0.30 +
        insider_scores[ticker]   * 0.30
    )
    idiosyncratic_risk[ticker] = round(score, 4)

print("종목별 고유 리스크 점수:")
print("-" * 40)
for ticker, score in idiosyncratic_risk.items():
    bar = "█" * int(score * 20)
    print(f"  {ticker:5s}: {score:.4f}  {bar}")

print("\n가장 위험한 종목:", max(idiosyncratic_risk, key=idiosyncratic_risk.get))
print("가장 안전한 종목:", min(idiosyncratic_risk, key=idiosyncratic_risk.get))

종목별 고유 리스크 점수:
----------------------------------------
  AAPL : 0.4293  ████████
  MSFT : 0.3831  ███████
  GOOGL: 0.2106  ████
  JPM  : 0.4332  ████████
  SOXX : 0.5000  ██████████

가장 위험한 종목: SOXX
가장 안전한 종목: GOOGL


In [14]:
# Step 5. 동적 자산 배분
# 포트폴리오 레벨 리스크 + 종목별 고유 리스크 통합

# 현재 포트폴리오 레벨 리스크
portfolio_risk = copula_scores["copula_risk_score"].iloc[-1]
current_regime = copula_scores["regime"].iloc[-1]

print(f"현재 포트폴리오 리스크: {portfolio_risk:.4f}")
print(f"현재 국면: {current_regime} ({'Normal' if current_regime==0 else 'Elevated' if current_regime==1 else 'Crisis'})")

def dynamic_allocation(portfolio_risk, idiosyncratic_risk, regime):
    """
    포트폴리오 리스크 + 고유 리스크 기반 동적 배분
    - 리스크 높을수록 비중 줄이고 현금 늘림
    """
    # 종목별 종합 리스크 (포트폴리오 50% + 고유 50%)
    combined_risk = {}
    for ticker in TICKERS:
        combined_risk[ticker] = round(
            portfolio_risk * 0.5 + idiosyncratic_risk[ticker] * 0.5, 4
        )

    # 리스크 역수로 기본 비중 계산
    inv_risk = {t: 1 / (r + 0.01) for t, r in combined_risk.items()}
    total_inv = sum(inv_risk.values())
    base_weights = {t: v / total_inv for t, v in inv_risk.items()}

    # 국면별 현금 비중 결정
    if regime == 0:    # Normal
        cash_ratio = max(0, portfolio_risk * 0.2)
    elif regime == 1:  # Elevated
        cash_ratio = max(0.1, portfolio_risk * 0.4)
    else:              # Crisis
        cash_ratio = max(0.3, portfolio_risk * 0.6)

    # 현금 제외한 나머지를 종목에 배분
    equity_ratio = 1 - cash_ratio
    final_weights = {t: w * equity_ratio for t, w in base_weights.items()}
    final_weights["CASH"] = cash_ratio

    return combined_risk, final_weights

combined_risk, weights = dynamic_allocation(
    portfolio_risk, idiosyncratic_risk, current_regime
)

print("\n종목별 종합 리스크:")
for ticker, risk in combined_risk.items():
    print(f"  {ticker}: {risk:.4f}")

print("\n동적 자산 배분 결과:")
print("-" * 40)
for asset, weight in sorted(weights.items(), key=lambda x: -x[1]):
    bar = "█" * int(weight * 30)
    print(f"  {asset:5s}: {weight*100:.1f}%  {bar}")

현재 포트폴리오 리스크: 0.2304
현재 국면: 1 (Elevated)

종목별 종합 리스크:
  AAPL: 0.3298
  MSFT: 0.3067
  GOOGL: 0.2205
  JPM: 0.3318
  SOXX: 0.3652

동적 자산 배분 결과:
----------------------------------------
  GOOGL: 24.4%  ███████
  MSFT : 17.7%  █████
  AAPL : 16.5%  ████
  JPM  : 16.4%  ████
  SOXX : 15.0%  ████
  CASH : 10.0%  ███


In [15]:
# 결과 저장
import json
from datetime import datetime

allocation_result = {
    "date": datetime.today().strftime("%Y-%m-%d"),
    "portfolio_risk": round(float(portfolio_risk), 4),
    "regime": int(current_regime),
    "idiosyncratic_risk": idiosyncratic_risk,
    "combined_risk": combined_risk,
    "weights": {k: round(v, 4) for k, v in weights.items()}
}

with open(f"{data_path}/allocation.json", "w") as f:
    json.dump(allocation_result, f, indent=2)

print("저장 완료: allocation.json")
print(json.dumps(allocation_result, indent=2))

저장 완료: allocation.json
{
  "date": "2026-07-08",
  "portfolio_risk": 0.2304,
  "regime": 1,
  "idiosyncratic_risk": {
    "AAPL": 0.4293,
    "MSFT": 0.3831,
    "GOOGL": 0.2106,
    "JPM": 0.4332,
    "SOXX": 0.5
  },
  "combined_risk": {
    "AAPL": 0.3298,
    "MSFT": 0.3067,
    "GOOGL": 0.2205,
    "JPM": 0.3318,
    "SOXX": 0.3652
  },
  "weights": {
    "AAPL": 0.1652,
    "MSFT": 0.1773,
    "GOOGL": 0.2436,
    "JPM": 0.1643,
    "SOXX": 0.1496,
    "CASH": 0.1
  }
}
